In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [2]:
# features = joblib.load('../predictors/hist/feats_fwd')

features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "elo_diff",
    "yellow_cards_rolling_5",
    "starts_rolling_3",
    "points_3_ewm",
    "xP_3_ewm",
    "minutes_rolling_3",
    "goals_scored_3_ewm",
    "assists_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_3_ewm",
    "expected_assists_3_ewm",
    "expected_goal_involvements_3_ewm",
    "interceptions_3_ewm",
    "blocks_3_ewm",
    "clearances_3_ewm",
    "tackles_3_ewm",
    "selected_by_percent",
    "transfers_in",
]


player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Forward"]

league_models = {}

for gw in range(21, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]
    # feats.remove('xP')
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.45,
        min_corr=0.15,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="FWD", random_seed=42
    )

    print(f"Features for FWD kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 21 
  dropping 'xP_3_ewm' (|corr| with xP=0.868) -- |corr|=0.897 with already-selected 'points_3_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.866) -- |corr|=0.937 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.801) -- |corr|=0.838 with already-selected 'points_3_ewm'
  dropping 'goals_scored_3_ewm' (|corr| with xP=0.785) -- |corr|=0.890 with already-selected 'points_3_ewm'
  dropping 'expected_goal_involvements_3_ewm' (|corr| with xP=0.776) -- |corr|=0.844 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.773) -- |corr|=0.810 with already-selected 'points_3_ewm'
  dropping 'expected_goals_3_ewm' (|corr| with xP=0.737) -- |corr|=0.805 with already-selected 'points_3_ewm'
  dropping 'transfers_in' (|corr| with xP=0.627) -- |corr|=0.692 with already-selected 'points_3_ewm'
  dropping 'selected_by_percent' (|corr| with xP=0.613) -- |corr|=0.622 with already-selected 'points_3_ewm'
  dropping 

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 41 seconds.
Initializing NUTS using jitter+adapt_diag...


  divergences: 0
Kept 3 / 5 features: ['elo_diff', 'yellow_cards_rolling_5', 'points_3_ewm']
Refitting on pruned feature set...


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


... (135 outputs left out)

Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 60 seconds.


  divergences: 0
Features for FWD kept in gw 38 -> ['opponent_difficulty', 'yellow_cards_rolling_5', 'points_3_ewm', 'interceptions_3_ewm', 'blocks_3_ewm']


In [3]:
joblib.dump(league_models, "./league_models_fwd")

['./league_models_fwd']